____
#### __Toy Datasets__
_Creating simple simulated datasets of varying levels of difficulty to see how well our PySR pipelines learn._

____

__A) perfect pISO imitation of the SPARC dataset (no noise)__

In [1]:
import numpy as np
import pandas as pd
from datawrangling import produce_SPARC_df
from mcmc_fit import v2_total, vdm2_pISO

def make_pISO_toy_df(df: pd.DataFrame, results: pd.DataFrame) -> pd.DataFrame:
    """
    Replace Vobs with the exact analytical pISO total velocity for each galaxy,
    using the fitted parameters from results. Baryonic components are unchanged.
    """
    rows = []
    for _, res in results.iterrows():
        gdf = df[df["galaxy"] == res["galaxy"]].copy()
        r = gdf["Rad_kpc"].values.astype(float)
        vgas = gdf["Vgas_km/s"].values.astype(float)
        vdisk = gdf["Vdisk_km/s"].values.astype(float)
        vbul = gdf["Vbul_km/s"].values.astype(float)

        v2 = v2_total(r, vgas, vdisk, vbul,
                      res["upsilon_disk"], res["upsilon_bulge"],
                      res["a"], res["b_kpc"], vdm2_pISO)
        gdf["Vobs_km/s"] = np.sqrt(np.maximum(v2, 0.0))
        gdf["upsilon_disk"] = res["upsilon_disk"]
        gdf["upsilon_bulge"] = res["upsilon_bulge"]
        rows.append(gdf)
    return pd.concat(rows, ignore_index=True)

df = produce_SPARC_df("data/SPARC", reduced_load=False)
results = pd.read_csv("outputs/fits/pISO.csv")
pISO_df = make_pISO_toy_df(df, results)
pISO_df.to_csv("data/toydatasets/pISO_noiseless.csv", index=False)

Taken data from 175 galaxies, making one DataFrame with 3391 rows.


___
__B) Adding noise__

Very crudely as a first pass, introduce multiplicative Gaussian noise onto the velocity values:
$$v \rightarrow v \cdot (1+\epsilon) \phantom{xxxxx} \epsilon \sim \mathcal{N}(0,\sigma^2)$$
Since the mean absolute error is c. 5\%, this corresponds to $\sigma \sim 0.05/0.8 \sim 0.06$. Therefore we apply three varying noise levels, $\sigma=0.02,0.05,0.1$ and see how the model behaves for each of those.

In [ ]:
def add_velocity_noise(df, sigma=0.05, seed=None):
    rng = np.random.default_rng(seed)
    noisy_df = df.copy()

    eps = rng.normal(loc=0.0, scale=sigma, size=len(df))

    noisy_df["errV_km/s"]  = sigma * noisy_df["Vobs_km/s"]
    noisy_df["Vobs_km/s"] *= (1 + eps)

    return noisy_df

In [ ]:
seed = 42
pISO_noisy_weak_df   = add_velocity_noise(pISO_df, sigma=0.02, seed=seed)
pISO_noisy_medium_df = add_velocity_noise(pISO_df, sigma=0.05, seed=seed)
pISO_noisy_strong_df = add_velocity_noise(pISO_df, sigma=0.1,  seed=seed)
# pISO_noisy_weak_df.to_csv("data/toydatasets/pISO_smallnoise.csv", index=False)
# pISO_noisy_medium_df.to_csv("data/toydatasets/pISO_midnoise.csv", index=False)
# pISO_noisy_strong_df.to_csv("data/toydatasets/pISO_strongnoise.csv", index=False)